In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
import json
import numpy as np
import random

load_dotenv()

# 로컬
ROOT = Path(os.environ["DATA_ROOT"])
HF_HOME = ROOT / ".hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

# 클라우드
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
# os.environ["HF_HOME"] = ".hf_cache"

from datasets import load_dataset
from sklearn.metrics import label_ranking_average_precision_score

In [ ]:
# Config
config = {
    "seed": 42,
    "num_labels": 188,
    "splits": ["val", "test"],      # val에서 튜닝, test에 1회 적용
    "anchor_tau": 0.5,              # 프로토콜 앵커 — 교체가 아니라 병기 대상
    "raw_ds": "ingyoun/patent-clean-text",
    "fields": ["invention_title", "ipc_main", "abstract", "claims"],    # 기록용
    "hf_cache": str(HF_HOME),
    "out_path": ROOT / "output",
}

# 로짓·라벨 연산만 하므로 모델 스펙은 tag/arch만 필요하다(체크포인트·토크나이저는 06_00 참조).
MODELS = [
    {"tag": "kobert-patent-baseline_len512", "arch": "kobert"},
    {"tag": "modernbert-patent-len512", "arch": "modernbert"},
    {"tag": "modernbert-patent-len8192", "arch": "modernbert"},
]

ANCHOR_TAG = "modernbert-patent-len8192"    # 곡선 예시로 쓸 모델(exp1)
TAU0 = config["anchor_tau"]
# global τ 탐색 그리드(거친 격자 → 최적 근방 0.01 세분)
GRID_COARSE = np.round(np.arange(0.05, 0.96, 0.05), 2)
# per-class τ 그리드. 세밀할수록 val-A 적합이 흔들리나, 채택 게이트가 걸러낸다.
GRID_CLASS = np.round(np.arange(0.05, 0.96, 0.01), 2)

In [ ]:
random.seed(config['seed'])
np.random.seed(config['seed'])

In [ ]:
class LogitLoader:
    def __init__(self, cache_dir: Path, tag: str, split: str):
        self.cache = cache_dir
        self.tag = tag
        self.split = split

    def get(self):
        fp = self.cache / f"logits_{self.tag}_{self.split}.npy"
        if not fp.exists():
            raise FileNotFoundError(f"로짓 캐시 없음: {fp} — 06_00으로 덤프 후 다운로드할 것")
        print(f"[load] {fp.name}")
        return np.load(fp)

In [ ]:
cache_dir = config["out_path"]

# prob[split][tag] — τ 연산은 전부 확률 위에서 한다.
prob = {s: {} for s in config["splits"]}
for s in config["splits"]:
    for d in MODELS:
        z = LogitLoader(cache_dir=cache_dir, tag=d["tag"], split=s).get()
        prob[s][d["tag"]] = 1.0 / (1.0 + np.exp(-z))

## 데이터 · 라벨

val·test 양쪽의 188차원 다중핫을 만든다. 클래스별 양성 표본 수를 함께 확인한다 — per-class τ 가드 설계의 전제(평탄 분포, 희소 꼬리 없음).

In [ ]:
Y = {}
for s in config["splits"]:
    ds = load_dataset(config["raw_ds"], split=s)
    doc_ids = json.loads((config["out_path"] / f"doc_ids_{s}.json").read_text(encoding="utf-8"))

    # 로짓 행 순서 == 데이터셋 행 순서 (06_00이 DataLoader(shuffle=False)로 보장한 축)
    assert ds["document_id"] == doc_ids, f"{s}: 로짓 행 순서와 데이터셋 행 순서가 다르다"

    y = np.zeros((len(ds), config["num_labels"]), dtype=bool)
    for i, ids in enumerate(ds["label_ids"]):
        y[i, ids] = True
    Y[s] = y
    for tag, P in prob[s].items():
        assert P.shape == y.shape, f"{s} {tag}"

    pos = y.sum(0)
    print(f"{s:>5}  n={len(ds):>7,}  k>=2 {(y.sum(1) >= 2).mean():.2%}  "
          f"클래스 양성 표본 min {pos.min()} / median {int(np.median(pos))} / max {pos.max()}")

## 지표 함수

τ는 스칼라(global) 또는 (188,) 벡터(per-class)를 동일하게 받는다.

In [ ]:
def _predict(P, tau):
    """tau는 스칼라(global) 또는 (C,) 벡터(per-class)."""
    return P >= (tau if np.isscalar(tau) else np.asarray(tau)[None, :])


def class_f1_from_pred(pred, Y):
    """클래스별 이진 F1. (C,)"""
    tp = (pred & Y).sum(0)
    fp = (pred & ~Y).sum(0)
    fn = (~pred & Y).sum(0)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)


def micro_f1_from_pred(pred, Y):
    tp = int((pred & Y).sum())
    fp = int((pred & ~Y).sum())
    fn = int((~pred & Y).sum())
    return 2 * tp / max(2 * tp + fp + fn, 1)


def sample_f1_from_pred(pred, Y):
    inter = (pred & Y).sum(1)
    return 2 * inter / np.maximum(pred.sum(1) + Y.sum(1), 1)


# τ를 직접 받는 래퍼 — 그리드 탐색용
def per_class_f1(P, Y, tau):
    return class_f1_from_pred(_predict(P, tau), Y)


def micro_f1(P, Y, tau):
    return micro_f1_from_pred(_predict(P, tau), Y)


def r_precision(P, Y):
    """문서별 상위 |정답| 예측의 정밀도. τ와 무관."""
    order = np.argsort(-P, axis=1)
    k = Y.sum(1)
    hits = np.array([Y[i, order[i, :k[i]]].sum() for i in range(len(Y))], dtype=float)
    return hits / np.maximum(k, 1)


def evaluate(P, Y, tau):
    """한 정책의 지표 묶음. empty rate는 argmax 강제 전 값(τ 자체의 성질)."""
    pred = _predict(P, tau)
    empty = pred.sum(1) == 0
    forced = pred.copy()
    if empty.any():
        forced[empty, P[empty].argmax(1)] = True
    return {
        "micro": round(micro_f1_from_pred(pred, Y), 4),
        "macro": round(float(class_f1_from_pred(pred, Y).mean()), 4),
        "sample": round(float(sample_f1_from_pred(pred, Y).mean()), 4),
        "empty_rate": round(float(empty.mean()), 6),
        "micro_argmax": round(micro_f1_from_pred(forced, Y), 4),
        "macro_argmax": round(float(class_f1_from_pred(forced, Y).mean()), 4),
        "sample_argmax": round(float(sample_f1_from_pred(forced, Y).mean()), 4),
    }

### verify — τ=0.5 앵커

τ=0.5 재계산치가 `output/total_metrics_{tag}.json`과 일치해야 이후 τ 비교가 같은 기준선 위에 선다.

In [ ]:
for d in MODELS:
    tag = d["tag"]
    r = evaluate(prob["test"][tag], Y["test"], TAU0)
    ssot = json.loads((config["out_path"] / f"total_metrics_{tag}.json").read_text(encoding="utf-8"))
    keep, arg = ssot["multilabel_f1"]["keep"], ssot["multilabel_f1"]["argmax"]
    pairs = [
        ("micro", r["micro"], keep["micro"]),
        ("macro", r["macro"], keep["macro"]),
        ("sample", r["sample"], keep["sample"]),
        ("micro+arg", r["micro_argmax"], arg["micro"]),
        ("sample+arg", r["sample_argmax"], arg["sample"]),
        ("empty", r["empty_rate"], ssot["empty_rate_tau_micro"]),
    ]
    print(tag)
    print("  " + "  ".join(f"{n} {a:.4f}/{b:.4f}" for n, a, b in pairs))
    for n, a, b in pairs:
        assert abs(a - b) < 1e-4, f"{tag} {n}: {a} vs {b}"

print("\nverify: τ=0.5 재계산치 == SSOT (3모델 × 6지표 일치)")

## global τ

val에서 micro·macro를 각각 최대화하는 단일 τ를 찾는다. 거친 격자(0.05 step) → 최적 근방 ±0.05를 0.01로 세분.

In [ ]:
def search_global(P, Y, objective):
    """거친 격자에서 최적을 잡고 근방 ±0.05를 0.01로 세분한다."""
    def score(t):
        return micro_f1(P, Y, t) if objective == "micro" else float(per_class_f1(P, Y, t).mean())

    best = max(GRID_COARSE, key=score)
    fine = np.round(np.arange(max(0.01, best - 0.05), min(0.99, best + 0.05) + 1e-9, 0.01), 2)
    best = max(fine, key=score)
    return float(best), float(score(best))


tau_global = {}
for d in MODELS:
    tag = d["tag"]
    Pv, Yv = prob["val"][tag], Y["val"]
    t_mi, s_mi = search_global(Pv, Yv, "micro")
    t_ma, s_ma = search_global(Pv, Yv, "macro")
    tau_global[tag] = {"micro": t_mi, "macro": t_ma,
                       "val_micro": round(s_mi, 4), "val_macro": round(s_ma, 4)}

print(f"{'tag':<34}{'τ(micro)':>10}{'val micro':>11}{'τ(macro)':>11}{'val macro':>11}")
for tag, r in tau_global.items():
    print(f"{tag:<34}{r['micro']:>10.2f}{r['val_micro']:>11.4f}"
          f"{r['macro']:>11.2f}{r['val_macro']:>11.4f}")

# τ에 따른 val 곡선 — 0.5가 최적에서 얼마나 떨어져 있는지
print(f"\n[{ANCHOR_TAG}] val micro/macro vs τ")
Pv, Yv = prob["val"][ANCHOR_TAG], Y["val"]
for t in GRID_COARSE[::2]:
    mark = "  ← τ=0.5" if abs(t - TAU0) < 1e-9 else ""
    print(f"  τ={t:.2f}  micro {micro_f1(Pv, Yv, t):.4f}  "
          f"macro {per_class_f1(Pv, Yv, t).mean():.4f}{mark}")

## per-class τ — held-out 채택 규칙

클래스별 τ는 클래스당 양성 표본 ~71개 위에서 고르므로 전 클래스가 과적합 위험을 동등하게 진다(희소 클래스 꼬리가 없는 평탄 분포 — `no-train-analysis.md` B).

가드는 표본 수 하한이 아니라 **일반화 여부를 직접 확인하는 형태**로 둔다: val을 문서 단위로 2분할해 val-A에서 τ_c를 적합하고, **val-B에서 global τ_macro보다 나은 클래스만 채택**한다. 탈락한 클래스는 global τ_macro를 쓴다.

**채택된 클래스의 τ는 val-A 적합값을 그대로 쓴다.** val 전체로 재적합하면 표본은 두 배가 되지만 val-B가 채택 판정과 τ 결정에 두 번 쓰여, 게이트의 선택 편향이 τ 값까지 옮겨온다. 아래 재적합 비교가 그 근거다.

In [ ]:
def tune_per_class(P, Y, grid):
    """클래스별 F1을 최대화하는 τ_c 벡터. (T, C) 스택에서 열별 argmax."""
    f1s = np.stack([per_class_f1(P, Y, t) for t in grid])
    return grid[f1s.argmax(0)]


# val 문서 단위 2분할 — 클래스별 τ의 일반화 여부를 held-out으로 확인하기 위한 분할.
rng = np.random.default_rng(config["seed"])
n_val = len(Y["val"])
perm = rng.permutation(n_val)
IDX_A, IDX_B = perm[: n_val // 2], perm[n_val // 2:]
print(f"val 2분할: A {len(IDX_A):,} / B {len(IDX_B):,}")

per_class = {}
for d in MODELS:
    tag = d["tag"]
    Pv, Yv = prob["val"][tag], Y["val"]
    t_glob = tau_global[tag]["macro"]        # per-class는 macro를 노리므로 macro 최적 global을 기준선으로 둔다

    tau_a = tune_per_class(Pv[IDX_A], Yv[IDX_A], GRID_CLASS)     # val-A 적합
    f1_b_class = per_class_f1(Pv[IDX_B], Yv[IDX_B], tau_a)       # val-B에서 클래스별 τ 성능
    f1_b_global = per_class_f1(Pv[IDX_B], Yv[IDX_B], t_glob)     # 같은 곳에서 global τ 성능
    adopt = f1_b_class > f1_b_global                             # 채택 게이트

    # 채택된 클래스는 val-A 적합값을 그대로 쓴다.
    # val 전체로 재적합하면 val-B가 채택 판정과 τ 결정에 두 번 쓰여 선택 편향이 τ 값까지 옮겨온다
    # (아래 no_refit / refit 비교가 근거 — 3모델 모두 재적합이 일반화를 악화시킨다).
    tau_vec = np.where(adopt, tau_a, t_glob)
    tau_refit = np.where(adopt, tune_per_class(Pv, Yv, GRID_CLASS), t_glob)

    per_class[tag] = {
        "tau_vec": tau_vec,
        "tau_vec_refit": tau_refit,
        "adopt": adopt,
        "n_adopted": int(adopt.sum()),
        "held_out_gain": round(float((f1_b_class - f1_b_global)[adopt].mean()) if adopt.any() else 0.0, 4),
    }

print(f"\n{'tag':<34}{'채택 클래스':>12}{'비율':>8}{'τ_c 중앙값':>12}{'τ_c 범위':>16}{'val-B 이득':>12}")
for d in MODELS:
    tag, r = d["tag"], per_class[d["tag"]]
    tv = r["tau_vec"][r["adopt"]]
    rng_txt = f"{tv.min():.2f}~{tv.max():.2f}" if len(tv) else "—"
    med = f"{np.median(tv):.2f}" if len(tv) else "—"
    print(f"{tag:<34}{r['n_adopted']:>8,}/188{r['n_adopted'] / 188:>8.1%}{med:>12}{rng_txt:>16}"
          f"{r['held_out_gain']:>+12.4f}")

# 재적합 유무 비교 — 프로토콜 선택의 근거를 산출물에 남긴다.
print(f"\n[재적합 비교] test macro 이득(τ=0.5 대비)과 일반화 격차")
refit_cmp = {}
for d in MODELS:
    tag = d["tag"]
    bv = evaluate(prob["val"][tag], Y["val"], TAU0)["macro"]
    bt = evaluate(prob["test"][tag], Y["test"], TAU0)["macro"]
    row = {}
    for name, tv in [("no_refit", per_class[tag]["tau_vec"]), ("refit", per_class[tag]["tau_vec_refit"])]:
        gv = evaluate(prob["val"][tag], Y["val"], tv)["macro"] - bv
        gt = evaluate(prob["test"][tag], Y["test"], tv)["macro"] - bt
        row[name] = {"val_gain": round(gv, 4), "test_gain": round(gt, 4), "gap": round(gt - gv, 4)}
    refit_cmp[tag] = row
    print(f"  {tag}")
    for name, v in row.items():
        print(f"    {name:<10}val {v['val_gain']:+.4f}  test {v['test_gain']:+.4f}  격차 {v['gap']:+.4f}")

## 정책 비교 매트릭스

모델별 {τ=0.5, global τ_micro, global τ_macro, per-class τ} × {micro, macro, sample-F1, empty rate}를 val·test 양쪽에서 낸다. `+arg`는 빈 예측 문서에 argmax 1개를 강제한 변형이다.

**val→test 일반화 격차** = (test 이득) − (val 이득), 둘 다 τ=0.5 대비. 음수가 클수록 val에 과적합한 정책이다 — per-class τ의 진단 지표.

In [ ]:
POLICIES = ["tau0", "global_micro", "global_macro", "per_class"]

report, gap = {}, {}
for d in MODELS:
    tag = d["tag"]
    taus = {
        "tau0": TAU0,
        "global_micro": tau_global[tag]["micro"],
        "global_macro": tau_global[tag]["macro"],
        "per_class": per_class[tag]["tau_vec"],
    }
    report[tag] = {
        pol: {s: evaluate(prob[s][tag], Y[s], taus[pol]) for s in config["splits"]}
        for pol in POLICIES
    }
    # val→test 일반화 격차 = (test 이득) − (val 이득), 앵커 τ=0.5 대비. 음수면 val 과적합.
    gap[tag] = {
        pol: {
            m: round((report[tag][pol]["test"][m] - report[tag]["tau0"]["test"][m])
                     - (report[tag][pol]["val"][m] - report[tag]["tau0"]["val"][m]), 4)
            for m in ["micro", "macro", "sample"]
        }
        for pol in POLICIES[1:]
    }

for d in MODELS:
    tag = d["tag"]
    print(tag)
    print(f"  {'정책':<14}{'split':>6}{'micro':>9}{'macro':>9}{'sample':>9}{'empty':>9}"
          f"{'micro+arg':>11}{'macro+arg':>11}{'sample+arg':>11}")
    for pol in POLICIES:
        for s in config["splits"]:
            r = report[tag][pol][s]
            print(f"  {pol:<14}{s:>6}{r['micro']:>9.4f}{r['macro']:>9.4f}{r['sample']:>9.4f}"
                  f"{r['empty_rate']:>9.2%}{r['micro_argmax']:>11.4f}{r['macro_argmax']:>11.4f}"
                  f"{r['sample_argmax']:>11.4f}")
    print("  test 이득(τ=0.5 대비) · 일반화 격차(test−val)")
    for pol in POLICIES[1:]:
        line = "  ".join(
            f"{m} {report[tag][pol]['test'][m] - report[tag]['tau0']['test'][m]:+.4f}"
            f" ({gap[tag][pol][m]:+.4f})" for m in ["micro", "macro", "sample"])
        print(f"    {pol:<14}{line}")
    print()

## 헤드룸 vs 추정 한계 — 오라클 · 학습 곡선

정책 비교에서 per-class τ가 test에서 이득을 못 내는데, 원인이 두 가지로 갈린다: **레버 자체가 없거나(헤드룸 부재)**, **레버는 있으나 val 표본으로 못 잡거나(추정 한계)**. 처방이 정반대이므로 갈라야 한다.

- **오라클**: test에서 직접 τ를 튜닝한 값. 실제로 쓸 수 없는 수치이며 **헤드룸의 상한**을 재는 용도다(06_01의 오라클-Lno와 같은 역할).
- **학습 곡선**: val을 subsample해 per-class τ를 적합하고 test 이득을 본다. 표본이 늘수록 결손이 줄면 추정 한계이고, 평평하면 헤드룸 부재다.

In [ ]:
oracle = {}
for d in MODELS:
    tag = d["tag"]
    Pt, Yt = prob["test"][tag], Y["test"]
    base = evaluate(Pt, Yt, TAU0)
    t_mi, _ = search_global(Pt, Yt, "micro")
    t_ma, _ = search_global(Pt, Yt, "macro")
    g_mi, g_ma = evaluate(Pt, Yt, t_mi), evaluate(Pt, Yt, t_ma)
    pc = evaluate(Pt, Yt, tune_per_class(Pt, Yt, GRID_CLASS))
    oracle[tag] = {
        "tau0_micro": base["micro"], "tau0_macro": base["macro"],
        "global_oracle_micro": g_mi["micro"], "global_oracle_macro": g_ma["macro"],
        "per_class_oracle_micro": pc["micro"], "per_class_oracle_macro": pc["macro"],
        "headroom_global_macro": round(g_ma["macro"] - base["macro"], 4),
        "headroom_per_class_macro": round(pc["macro"] - base["macro"], 4),
    }

print(f"{'tag':<34}{'τ=0.5':>9}{'global 오라클':>15}{'per-class 오라클':>18}"
      f"{'헤드룸(global)':>16}{'헤드룸(per-class)':>19}")
for tag, r in oracle.items():
    print(f"{tag:<34}{r['tau0_macro']:>9.4f}{r['global_oracle_macro']:>15.4f}"
          f"{r['per_class_oracle_macro']:>18.4f}{r['headroom_global_macro']:>+16.4f}"
          f"{r['headroom_per_class_macro']:>+19.4f}")
print("\nglobal τ는 완벽히 골라도 얻을 것이 없고, per-class τ는 헤드룸이 크다 — 둘은 성격이 다르다.")

In [ ]:
LC_FRACS = [0.25, 0.5, 1.0]
LC_REPS = 5

lc_rng = np.random.default_rng(config["seed"])
learning_curve = {}
for d in MODELS:
    tag = d["tag"]
    Pv, Yv = prob["val"][tag], Y["val"]
    Pt, Yt = prob["test"][tag], Y["test"]
    base = evaluate(Pt, Yt, TAU0)["macro"]
    rows = []
    for frac in LC_FRACS:
        reps = LC_REPS if frac < 1.0 else 1          # val 전체는 표본이 하나뿐이라 반복 불필요
        gains, n_pos = [], []
        for _ in range(reps):
            idx = (lc_rng.choice(len(Yv), int(len(Yv) * frac), replace=False)
                   if frac < 1.0 else np.arange(len(Yv)))
            gains.append(evaluate(Pt, Yt, tune_per_class(Pv[idx], Yv[idx], GRID_CLASS))["macro"] - base)
            n_pos.append(int(np.median(Yv[idx].sum(0))))
        rows.append({
            "val_frac": frac,
            "median_pos_per_class": int(np.median(n_pos)),
            "test_macro_gain": round(float(np.mean(gains)), 4),
            "sd": round(float(np.std(gains)), 4),
        })
    learning_curve[tag] = rows

print("val 표본을 늘리면 test 이득이 오르는가 — 게이트 없이 per-class τ만 적합(추정 품질의 순수 신호)")
print(f"{'tag':<34}" + "".join(f"{f'{int(f*100)}%':>16}" for f in LC_FRACS))
for d in MODELS:
    tag = d["tag"]
    cells = "".join(f"{r['test_macro_gain']:>+11.4f}(~{r['median_pos_per_class']:>3})"
                    for r in learning_curve[tag])
    print(f"{tag:<34}{cells}")
print("  괄호는 클래스당 양성 표본 중앙값. 결손이 단조로 줄면 헤드룸 부재가 아니라 추정 한계다.")

## 랭킹 지표 불변 확인

LRAP·R-Precision은 τ와 무관하다. τ 정책을 바꿔도 값이 움직이지 않아야 하며, 기존 SSOT와도 일치해야 한다.

In [ ]:
ranking = {}
for d in MODELS:
    tag = d["tag"]
    P = prob["test"][tag]
    lrap = float(label_ranking_average_precision_score(Y["test"].astype(int), P))
    rp = float(r_precision(P, Y["test"]).mean())
    ssot = json.loads((config["out_path"] / f"total_metrics_{tag}.json").read_text(encoding="utf-8"))
    ranking[tag] = {
        "lrap": round(lrap, 4),
        "r_precision": round(rp, 4),
        "lrap_ssot": round(ssot["ranking"]["lrap"], 4),
        "r_precision_ssot": round(ssot["ranking"]["r_precision"], 4),
    }
    assert abs(lrap - ssot["ranking"]["lrap"]) < 1e-4, tag
    assert abs(rp - ssot["ranking"]["r_precision"]) < 1e-4, tag

print("랭킹 지표는 τ와 무관하므로 어떤 정책에서도 불변이다 — 파이프라인 sanity check.")
print(f"{'tag':<34}{'LRAP':>9}{'R-Prec':>9}")
for tag, r in ranking.items():
    print(f"{tag:<34}{r['lrap']:>9.4f}{r['r_precision']:>9.4f}")
print("\nverify: LRAP·R-Precision == SSOT (3/3 일치)")

## 저장

In [ ]:
meta = {
    "splits": {s: int(len(Y[s])) for s in config["splits"]},
    "anchor_tau": TAU0,
    "num_labels": config["num_labels"],
    "fields": config["fields"],
    "raw_ds": config["raw_ds"],
    "policy": {
        "tuning_split": "val",
        "applied_split": "test",
        "global_grid": "0.05~0.95 step 0.05 → 최적 근방 ±0.05를 step 0.01로 세분",
        "per_class_grid": "0.05~0.95 step 0.01",
        "per_class_guard": "val 문서 단위 2분할 — val-A에서 τ_c 적합, val-B에서 global τ_macro 대비 "
                           "클래스 F1 개선이 확인된 클래스만 채택, 나머지는 global τ_macro fallback. "
                           "채택된 클래스의 τ는 val-A 적합값을 유지한다(재적합하지 않는다).",
        "note": "3모델에 동일 절차를 적용한다(τ 값 자체는 모델별로 달라진다).",
    },
    "oracle_note": "오라클은 test에서 직접 τ를 튜닝한 값이라 적용 대상이 아니다 — 헤드룸 상한 측정용.",
    "learning_curve_note": f"게이트 없이 per-class τ만 적합. {LC_FRACS} 비율, "
                           f"{LC_REPS}회 반복 평균(val 전체는 1회).",
}

for d in MODELS:
    tag = d["tag"]
    result = {
        **meta,
        "tag": tag,
        "arch": d["arch"],
        "tau": {
            "global_micro": tau_global[tag]["micro"],
            "global_macro": tau_global[tag]["macro"],
            "per_class_adopted": int(per_class[tag]["n_adopted"]),
            "per_class_vector": [round(float(t), 2) for t in per_class[tag]["tau_vec"]],
        },
        "metrics": {
            pol: {s: report[tag][pol][s] for s in config["splits"]}
            for pol in POLICIES
        },
        "generalization_gap": gap[tag],
        "refit_comparison": refit_cmp[tag],
        "oracle": oracle[tag],
        "learning_curve": learning_curve[tag],
        "ranking_invariance": ranking[tag],
    }
    fp = config["out_path"] / f"threshold_{tag}.json"
    fp.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"saved: {fp}")